# Datenjournalismus-Pipeline

Dieses Notebook bündelt die gesamte Recherche-Pipeline in einem Ablauf. Die eigentliche Logik liegt in den Modulen unter `ddj_scripts/`, sodass ihr sie auch in eigenen Notebooks wiederverwenden könnt.

Inhaltlich passiert Folgendes:

1. Ihr gebt eurer Recherche einen Namen (Variable `thema`).
2. Ihr definiert Suchbegriffe.
3. Optional: Ihr schränkt die Suche über ein Präfix des Regionalschlüssels geografisch ein. Standardmäßig werden alle auslesbaren Ratsinformationssysteme (RIS) durchsucht.
4. Die Metadaten zu allen Kommunen ("Entities") werden geladen, um Treffer Kommunen zuordnen zu können.
5. Die Suche wird ausgeführt, die Treffer werden zu Sitzungen bzw. Vorgängen gruppiert und ausgewertet.

**Alles, was ihr anpassen müsst, steht in Abschnitt 1.** Danach könnt ihr alle Zellen der Reihe nach ausführen ("Run All") und bekommt:

- eine Rohdaten-CSV mit allen Treffern (`data/raw/`)
- eine aufbereitete CSV mit gruppierten Sitzungen/Vorgängen (`data/raw/`)
- ausgewertete CSVs sowie Grafiken zu Verteilung, Top-Kommunen und zeitlichem Verlauf (`data/processed/`)

Voraussetzung: `.env` ist ausgefüllt (siehe readme.md) und die venv ist als Kernel ausgewählt.

Dokumentation zu Poliscope inklusive API-Doku: [Doku](https://docs.poliscope.de/)

In [ ]:
import sys
from datetime import datetime as dt
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import altair as alt

from setup import *
from ddj_scripts.entities import load_entities
from ddj_scripts.search import fetch_search_results
from ddj_scripts.grouping import add_entity_name, get_unique_meetings
from ddj_scripts.analysis import (
    altair_theme,
    build_entity_counts,
    build_histogram,
    build_weekly_matches,
    get_text_color,
    load_theme,
)

datestring = dt.now().strftime("%Y-%m-%d")

## 1. Thema und Suchbegriffe eingeben

- `thema`: kurzer Name ohne Leerzeichen/Umlaute, wird für Dateinamen verwendet (z. B. `hitzeschutz`)
- `search_terms`: Liste an Suchbegriffen, die mit ODER verknüpft werden
- `entity_filter`: optional. Mit einem Präfix des [Regionalschlüssels (ARS)](https://www.destatis.de/DE/Themen/Laender-Regionen/Regionales/Gemeindeverzeichnis/Glossar/regionalschluessel.html) (z. B. `"03"` für Niedersachsen, `"03401"` für Delmenhorst) könnt ihr die Suche auf ein Bundesland/eine Region einschränken. Bei `None` wird bundesweit gesucht.

In [ ]:
thema = "waermeplanung"
search_terms = ["Wärmeplanung", "Wärmeplan", "Fernwärme", "Fernwärmenetz", "Wärmenetz", "Wärmeversorgung", "Wärmeversorgungskonzept", "Wärmeversorgungskonzepte", "Wärmeplanungsgesetz"]
entity_filter = None  # z.B. "03" für nur Niedersachsen, None wenn ganz Deutschland

## 2. Entities laden

Wird für die Zuordnung von Treffern zu Kommunen benötigt. Falls `data/metadata/all_entities.csv` bereits existiert, wird die Datei aus dem Cache geladen, statt erneut von der API zu laden.

In [ ]:
entities_path = Path("./data/metadata/all_entities.csv")
entities_df = load_entities(entities_path)

## 3. Suche ausführen

Ruft alle Treffer (Chunks) für die Suchbegriffe aus Abschnitt 1 ab.

In [ ]:
items_df = fetch_search_results(
    search_terms,
    entity_filter=entity_filter,
    thema=thema,
    raw_dir="./data/raw",
    datestring=datestring,
)

## 4. Treffer zu Sitzungen/Vorgängen gruppieren

Die Suche liefert einzelne Textabschnitte ("Chunks"). Mehrere Chunks können zur selben Sitzung oder zum selben Vorgang gehören und werden hier zusammengefasst.

- Ein **Meeting** ist eine einzelne Sitzung.
- Ein **Proposal** ist ein Vorgang, dem mehrere Sitzungen zugeordnet sein können (nicht jedes RIS pflegt diese Zuordnung).

In [ ]:
grouped_matches = get_unique_meetings(items_df)
grouped_matches = add_entity_name(grouped_matches, entities_df)

grouped_path = Path(f"./data/raw/{datestring}_{thema}_grouped_matches.csv")
grouped_matches.to_csv(grouped_path, index=False)
print(f"{len(grouped_matches)} gruppierte Treffer gespeichert unter {grouped_path}")
grouped_matches.head()

## 5. Auswertung & Grafiken

Lädt das Poliscope-Farbschema und erstellt Standard-Auswertungen: Verteilung der Treffer pro Kommune, Top-20-Kommunen und zeitlicher Verlauf.

In [ ]:
theme_data = load_theme("data/styles/poliscope_theme.json")


@alt.theme.register("poliscope_theme", enable=True)
def poliscopetheme():
    return altair_theme(theme_data)


text_color = get_text_color(theme_data)
df = grouped_matches

In [ ]:
# Verteilung der Treffer pro Kommune
entity_counts = build_entity_counts(df)
entity_counts.to_csv(f"./data/processed/{datestring}_entity_counts_{thema}.csv", index=False)

histogram_df = build_histogram(entity_counts)
histogram_df.to_csv(f"./data/processed/{datestring}_histogram_entity_counts_{thema}.csv", index=False)

mean_matches_per_entity = entity_counts["match_count"].mean()

histogram = (
    alt.Chart(entity_counts)
    .mark_bar(color="#00373a")
    .encode(
        x=alt.X("match_count:Q", bin=alt.Bin(step=5), title="Treffer pro Kommune"),
        y=alt.Y("count():Q", title="Anzahl Kommunen"),
        tooltip=["match_count:Q", "count():Q"],
    )
    .properties(width=800, height=300, title=f"Verteilung der Treffer pro Kommune (Mittelwert: {mean_matches_per_entity:.2f})")
    .configure_axis(labelColor=text_color, titleColor=text_color, grid=True)
    .configure_title(color=text_color)
)

histogram

In [ ]:
# Top 20 Städte nach Trefferanzahl, aufgeschlüsselt nach Meeting/Proposal
stack_col = "groupType"
plot_df = (
    df.assign(city=df["entityName"].fillna("Unknown"))
    .groupby(["city", stack_col], dropna=False)
    .size()
    .reset_index(name="matches")
)

city_totals = plot_df.groupby("city")["matches"].sum().sort_values(ascending=False)
city_totals.to_csv(f"./data/processed/{datestring}_city_totals_{thema}.csv", index=True)

top_cities = city_totals.head(20).index.tolist()
plot_df = plot_df[plot_df["city"].isin(top_cities)].copy()

category_colors = ["#306969", "#4FB0B0"]

chart = (
    alt.Chart(plot_df)
    .mark_bar(size=15)
    .encode(
        y=alt.Y("city:N", title=None, sort=top_cities),
        x=alt.X("matches:Q", title="Anzahl Treffer", scale=alt.Scale(zero=True)),
        color=alt.Color(f"{stack_col}:N", title="Kategorie", scale=alt.Scale(range=category_colors)),
        tooltip=["city:N", f"{stack_col}:N", "matches:Q"],
    )
    .properties(width=800, height=400, title="Top 20 Städte nach Trefferanzahl")
    .configure_axis(labelColor=text_color, titleColor=text_color, grid=True)
    .configure_title(color=text_color)
    .configure_legend(labelColor=text_color, titleColor=text_color)
)

chart

In [ ]:
# Treffer über Zeit. Wir betrachten nur die Zeit ab Mitte 2023, weil Poliscope erst ab etwa
# Anfang 2024 einen nahezu vollständigen Datensatz bietet (ältere Sitzungen tauchen nur vereinzelt auf).
weekly = build_weekly_matches(df)
weekly.to_csv(f"./data/processed/{datestring}_weekly_matches_{thema}.csv", index=False)

weekly_chart = (
    alt.Chart(weekly)
    .mark_bar(size=2)
    .encode(
        x=alt.X(
            "week:T",
            title="Woche",
            axis=alt.Axis(format="%Y-%m-%d", labelExpr="datum.value % 3 == 0 ? timeFormat(datum.value, '%Y-%m-%d') : ''"),
        ),
        y=alt.Y("matches:Q", title="Anzahl Treffer", scale=alt.Scale(zero=True)),
        color=alt.ColorValue("#00373a"),
        tooltip=["week_label:N", "matches:Q"],
    )
    .properties(width=800, height=300, title=f"{thema}: Treffer pro Woche")
    .configure_axis(labelColor=text_color, titleColor=text_color, grid=True)
    .configure_title(color=text_color)
)

weekly_chart

## Fertig!

Alle Ergebnisse liegen jetzt in `data/raw/` (Rohdaten & gruppierte Treffer) und `data/processed/` (Auswertungen). Für tiefergehende, individuelle Recherche könnt ihr die Funktionen aus `ddj_scripts/` in eigenen Notebooks nutzen. Gespeicherte Treffer-CSVs ladet ihr am besten mit `ddj_scripts.search.load_search_results`, damit die Kommunen-IDs ihre führenden Nullen behalten.